### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="chronic_kidney_disease",
    dataset_year="2015",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5G020",
    download_description="""
We get the _full version of the dataset from UCI.

wget https://archive.ics.uci.edu/static/public/336/chronic+kidney+disease.zip && unzip chronic+kidney+disease.zip && rm chronic+kidney+disease.zip && unrar x Chronic_Kidney_Disease.rar
# At this point I had unrar the file manually as there is no easily non-sudo installable tool for unpacking .rar files as it seems
mkdir -p local-data-warehouse/chronic_kidney_disease && mv chronic_kidney_disease_full.arff local-data-warehouse/chronic_kidney_disease/
""",
    # References
    academic_reference_bibtex="""@misc{Rubini2015chronickidney,
  author       = {Rubini, L., Soundarapandian, P., and Eswaran, P.},
  title        = {{Chronic Kidney Disease}},
  year         = {2015},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5G020}
}
""",
    academic_reference_bibtex_key="Rubini2015chronickidney",
    licence="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- The .arff file has some formatting issues (commas at the end of lines) that we fix in preprocessing. This might point to some weird manual data editing that at some point corrupted the data. But as the data is cited and used a lot, I would assume that the formatting issues are just that and do not reflect deeper data quality issues.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
)

## Preprocessing

In [2]:
from pathlib import Path
import pandas as pd
import re

def load_arff_no_skips(path: str | Path) -> tuple[pd.DataFrame, dict]:
    path = Path(path)
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

    relation = None
    attributes: list[tuple[str, str]] = []
    data_started = False

    # --- parse header ---
    for raw in lines:
        line = raw.strip()
        if not line or line.startswith("%"):
            continue
        low = line.lower()
        if low.startswith("@relation"):
            relation = line.split(None, 1)[1].strip().strip("'\"")
        elif low.startswith("@attribute"):
            m = re.match(r"@attribute\s+('.*?'|\".*?\"|\S+)\s+(.+)$", line, flags=re.I)
            if not m:
                raise ValueError(f"Could not parse attribute line: {raw}")
            name = m.group(1).strip().strip("'\"")
            typ = m.group(2).strip()
            attributes.append((name, typ))
        elif low.startswith("@data"):
            data_started = True
            break

    if not data_started or not attributes:
        raise ValueError("Failed to find a valid ARFF header with @attributes and @data.")

    n_cols = len(attributes)
    col_names = [a[0] for a in attributes]
    numeric_cols = [name for name, typ in attributes if typ.lower() == "numeric"]

    # --- parse data ---
    rows = []
    in_data = False
    for lineno, raw in enumerate(lines, start=1):
        line = raw.strip()
        if not line or line.startswith("%"):
            continue
        low = line.lower()
        if low.startswith("@data"):
            in_data = True
            continue
        if not in_data:
            continue

        # normalize spaces around commas (but keep empty fields)
        line = re.sub(r"\s*,\s*", ",", line).rstrip(",")  # trailing commas are just junk

        parts = line.split(",")  # preserves middle empties
        # map "?" to NA; keep "" as "" for now so we can detect and repair overlong rows
        parts = [pd.NA if p == "?" else p for p in parts]

        # --- REPAIR: if too many fields, drop empty tokens created by ",,"
        if len(parts) > n_cols:
            # indices where token is empty string (created by ",,")
            empty_idxs = [i for i, p in enumerate(parts) if p == ""]
            # remove empties until width matches
            while len(parts) > n_cols and empty_idxs:
                idx = empty_idxs.pop(0)
                parts.pop(idx)
                # recompute remaining empty indices after removal
                empty_idxs = [i for i, p in enumerate(parts) if p == ""]
            # if still too many, it's not just ",," -> hard error
            if len(parts) > n_cols:
                raise ValueError(
                    f"Line {lineno}: too many fields ({len(parts)}), expected {n_cols}.\n"
                    f"Raw: {raw}"
                )

        # now convert remaining "" to NA (true missing)
        parts = [pd.NA if str(p) == "" else p for p in parts]

        # pad if too short (do not skip)
        if len(parts) < n_cols:
            parts = parts + [pd.NA] * (n_cols - len(parts))

        rows.append(parts)

    df = pd.DataFrame(rows, columns=col_names)

    # coerce numerics
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    meta = {"relation": relation, "attributes": attributes, "n_rows": len(df), "n_cols": n_cols}
    return df, meta


# ---- usage ----
df, meta = load_arff_no_skips(str(dataset_mold.path / "chronic_kidney_disease_full.arff"))

In [3]:
as_cat_type = ["class", "rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"]
df[as_cat_type] = df[as_cat_type].astype("category")
for c in ["sg", "al", "su"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 400
Columns: 25
Use sampling: False (sample size: 400)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['bgr', 'bu', 'hemo', 'wbcc', 'sc', 'age', 'rbcc', 'pcv', 'pot', 'sod']
Rows remaining as candidates after top-10 filter: 0 (of 400)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,19.0,70.0,1.020,0.0,0.0,NaN,normal,notpresent,notpresent,NaN,NaN,NaN,NaN,NaN,11.5,NaN,6900.0,NaN,no,no,no,good,no,no,ckd
1,47.0,80.0,NaN,NaN,NaN,NaN,NaN,notpresent,notpresent,93.0,33.0,0.9,144.0,4.5,13.3,52.0,8100.0,5.2,no,no,no,good,no,no,notckd
2,60.0,100.0,1.020,2.0,0.0,abnormal,abnormal,notpresent,notpresent,140.0,55.0,2.5,NaN,NaN,10.1,29.0,NaN,NaN,yes,no,no,poor,no,no,ckd
3,59.0,100.0,1.015,4.0,2.0,normal,normal,notpresent,notpresent,255.0,132.0,12.8,135.0,5.7,7.3,20.0,9800.0,3.9,yes,yes,yes,good,no,yes,ckd
4,73.0,100.0,1.010,3.0,2.0,abnormal,abnormal,present,notpresent,295.0,90.0,5.6,140.0,2.9,9.2,30.0,7000.0,3.2,yes,yes,yes,poor,no,no,ckd


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,rbc,category,152.0,38.00,2.0,"normal, abnormal"
1,pc,category,65.0,16.25,2.0,"normal, abnormal"
2,pcc,category,4.0,1.00,2.0,"notpresent, present"
3,ba,category,4.0,1.00,2.0,"notpresent, present"
4,htn,category,2.0,0.50,2.0,"no, yes"
5,dm,category,2.0,0.50,2.0,"no, yes"
6,cad,category,2.0,0.50,2.0,"no, yes"
7,appet,category,1.0,0.25,2.0,"good, poor"
8,pe,category,1.0,0.25,2.0,"no, yes"
9,ane,category,1.0,0.25,2.0,"no, yes"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,391.0,51.483376,17.169714,2.000,90.000
bp,388.0,76.469072,13.683637,50.000,180.000
sg,353.0,1.017408,0.005717,1.005,1.025
al,354.0,1.016949,1.352679,0.000,5.000
su,351.0,0.450142,1.099191,0.000,5.000
bgr,356.0,148.036517,79.281714,22.000,490.000
bu,381.0,57.425722,50.503006,1.500,391.000
sc,383.0,3.072454,5.741126,0.400,76.000
sod,313.0,137.528754,10.408752,4.500,163.000
pot,312.0,4.627244,3.193904,2.500,47.000


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                          
ane    1             no    339  84.75
       2            yes     60  15.00
       3           <NA>      1   0.25
appet  1           good    317  79.25
       2           poor     82  20.50
       3           <NA>      1   0.25
ba     1     notpresent    374  93.50
       2        present     22   5.50
       3           <NA>      4   1.00
cad    1             no    364  91.00
       2            yes     34   8.50
       3           <NA>      2   0.50
class  1            ckd    250  62.50
       2         notckd    150  37.50
dm     1             no    261  65.25
       2            yes    137  34.25
       3           <NA>      2   0.50
htn    1             no    251  62.75
       2            yes    147  36.75
       3           <NA>      2   0.50
pc     1         normal    259  64.75
       2       abnormal     76  19.00
       3           <NA>     65  16.25
pcc    1     notpresent    354  88.50
       2        present     42  10.50
       3           <NA>      4   1.00
pe     1             no    323  80.75
       2            yes     76  19.00
       3           <NA>      1   0.25
rbc    1         normal    201  50.25
       2           <NA>    152  38.00
       3       abnormal     47  11.75

In [9]:
# Target Distribution
target_df

,count,pct
class,,
ckd,250,62.5
notckd,150,37.5


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7c87-fc05-74e7-a87f-b4ecb8f7c075
b3aeb75b1b30fe0b58192416d92c1421bb6c3688c7c796b48bdab6c4c19aa2a8
